In [6]:
import random
from textwrap import dedent
from typing import Dict, List
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer

# Function to create a test prompt from the dataset
def create_test_prompt(data_row):
    prompt = dedent(
        f"""
    {data_row["Prompt"]}

    Essay:

    ```
    {data_row["Essay"]}
    ```
    """
    )
    messages = [
        {
            "role": "system",
            "content": "You are an expert essay evaluator. Provide feedback to college essays by highlighting strengths, suggesting improvements, and scoring from 1 to 9 (9 = exceptional).",
        },
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

In [7]:
dataset = load_dataset(
    "json",
    data_files={"train": "train.json", "validation": "val.json", "test": "test.json"},
)
dataset

DatasetDict({
    train: Dataset({
        features: ['File', 'Prompt', 'Essay', 'Feedback', 'Score', 'text', 'tokenCount'],
        num_rows: 2400
    })
    validation: Dataset({
        features: ['File', 'Prompt', 'Essay', 'Feedback', 'Score', 'text', 'tokenCount'],
        num_rows: 500
    })
    test: Dataset({
        features: ['File', 'Prompt', 'Essay', 'Feedback', 'Score', 'text', 'tokenCount'],
        num_rows: 100
    })
})

In [8]:
MODEL_NAME = "jinwkim/Llama-3-8B-Instruct-Essay-Evaluation"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=quantization_config, device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.90s/it]


In [18]:
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    return_full_text=False,
)

Device set to use cuda:0


In [19]:
row = dataset["test"][0]
prompt = create_test_prompt(row)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert essay evaluator. Provide feedback to college essays by highlighting strengths, suggesting improvements, and scoring from 1 to 9 (9 = exceptional).<|eot_id|><|start_header_id|>user<|end_header_id|>

Share an essay on any topic of your choice. It can be one you've already written, one that responds to a different prompt, or one of your own design. (650 words max)

Essay:

```
After the usual hour-long piano lesson to satisfy my mom, we’d pull into my favorite place on Earth: Michaels. Drawn by the strange charm of its scent of cloying pine and cinnamon, I’d walk in, eyes lighting up as I scanned through never ending rows of creative freedom—a maze that I would allow myself to get lost in. In my youth, I refused to leave Michaels without a gallon of Elmer’s glue from the glue section. Carefully combined with the right ratio of borax to water, the entire gallon would be turned into a delightfully sticky mess: sl

In [ ]:
outputs = pipe(prompt)
print(f"Human feedback: {row['Feedback']}")
print(f"AI feedback: {outputs[0]['generated_text']}")